In [1]:
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta
from tqdm import tqdm

# Load the base tables
uscities = pd.read_csv('C:/Users/10741867/OneDrive - LTIMindtree/Documents/Datasets/Solution Datasets/Solution 1 - PNC Quote Conversion and Prioritization/Datasets/Regenerated Datasets/uscities.csv')
auto_make_model = pd.read_csv('C:/Users/10741867/OneDrive - LTIMindtree/Documents/Datasets/Solution Datasets/Solution 1 - PNC Quote Conversion and Prioritization/Datasets/Regenerated Datasets/AUTO_MAKE_MODEL.csv')
auto_insurance_agent_master = pd.read_csv('C:/Users/10741867/OneDrive - LTIMindtree/Documents/Datasets/Solution Datasets/Solution 1 - PNC Quote Conversion and Prioritization/Datasets/Regenerated Datasets/AUTO_INSURANCE_AGENT_MASTER.csv')
auto_insurance_policy_master = pd.read_csv('C:/Users/10741867/OneDrive - LTIMindtree/Documents/Datasets/Solution Datasets/Solution 1 - PNC Quote Conversion and Prioritization/Datasets/Regenerated Datasets/AUTO_INSURANCE_POLICY_MASTER.csv')

# Convert POLICY_BIND_DATE to datetime
auto_insurance_policy_master['POLICY_BIND_DATE'] = pd.to_datetime(auto_insurance_policy_master['POLICY_BIND_DATE'])

# Generate synthetic data for AUTO_INSURANCE_CLAIMS_MASTER_BASE
def generate_claims_data(policy_master, num_records):
    claims_data = []
    for _ in tqdm(range(num_records), desc="Generating Claims Data"):
        # CLAIMS_NUMBER
        claims_number = 'P' + str(random.randint(10000000, 99999999))
        
        # INCIDENT_DATE
        incident_date = datetime.strptime('{} {}'.format(random.randint(1, 366), random.randint(2015, 2024)), '%j %Y').date()
        
        # POLICY_NUMBER
        valid_policies = policy_master[policy_master['POLICY_BIND_DATE'] < pd.Timestamp(incident_date)]
        if valid_policies.empty:
            continue
        policy_number = valid_policies.sample(1)['POLICY_NUMBER'].values[0]
        
        # CLAIM_REPORTING_DATE
        claim_reporting_date = incident_date + timedelta(days=random.randint(0, 15))
        
        # INCIDENT_TYPE
        incident_type_probs = {
            "Vehicle Theft": 0.21,
            "Parked Car": 0.17,
            "Single Vehicle Collision": 0.45,
            "Multi Vehicle Collision": 0.17
        }
        incident_type = random.choices(list(incident_type_probs.keys()), list(incident_type_probs.values()))[0]
        
        # COLLISION_TYPE
        if incident_type == "Vehicle Theft":
            collision_type = "Details not Available"
        elif incident_type == "Parked Car":
            collision_type = "Side Collision"
        else:
            collision_type_probs = {
                "Rear Collision": 0.45,
                "Front Collision": 0.55
            }
            collision_type = random.choices(list(collision_type_probs.keys()), list(collision_type_probs.values()))[0]
        
        # LOSS_TYPE
        if incident_type == "Vehicle Theft":
            loss_type = "Total Loss"
        elif incident_type == "Parked Car":
            loss_type = "Trivial Damage"
        elif incident_type == "Single Vehicle Collision":
            loss_type_probs = {
                "Trivial Damage": 0.35,
                "Minor Damage": 0.40,
                "Major Damage": 0.25
            }
            loss_type = random.choices(list(loss_type_probs.keys()), list(loss_type_probs.values()))[0]
        else:
            loss_type_probs = {
                "Major Damage": 0.65,
                "Total Loss": 0.35
            }
            loss_type = random.choices(list(loss_type_probs.keys()), list(loss_type_probs.values()))[0]
        
        # AUTHORITIES_CONTACTED
        if incident_type == "Vehicle Theft":
            authorities_contacted = "Police"
        elif incident_type == "Multi Vehicle Collision":
            authorities_contacted_probs = {
                "Police": 0.65,
                "Fire": 0.15,
                "Ambulance": 0.20
            }
            authorities_contacted = random.choices(list(authorities_contacted_probs.keys()), list(authorities_contacted_probs.values()))[0]
        elif incident_type == "Single Vehicle Collision":
            authorities_contacted_probs = {
                "Police": 0.35,
                "Fire": 0.05,
                "Ambulance": 0.20,
                None: 0.40
            }
            authorities_contacted = random.choices(list(authorities_contacted_probs.keys()), list(authorities_contacted_probs.values()))[0]
        else:
            authorities_contacted = None
        
        # INCIDENT_STATE, INCIDENT_CITY, INCIDENT_COUNTY
        policy_details = policy_master[policy_master['POLICY_NUMBER'] == policy_number].iloc[0]
        incident_state = policy_details['POLICY_STATE']
        incident_city = policy_details['POLICY_CITY']
        incident_county = policy_details['POLICY_COUNTY']
        
        # INCIDENT_TIME_OF_DAY
        if incident_type == "Vehicle Theft":
            time_of_day_probs = {
                "Early Morning Hours": 0.35,
                "Morning to Noon": 0.20,
                "Afternoon Hours": 0.15,
                "Night Time": 0.30
            }
        elif incident_type == "Parked Car":
            time_of_day_probs = {
                "Early Morning Hours": 0.05,
                "Morning to Noon": 0.35,
                "Afternoon Hours": 0.30,
                "Night Time": 0.30
            }
        elif incident_type == "Single Vehicle Collision":
            time_of_day_probs = {
                "Early Morning Hours": 0.20,
                "Morning to Noon": 0.25,
                "Afternoon Hours": 0.15,
                "Night Time": 0.40
            }
        else:
            time_of_day_probs = {
                "Early Morning Hours": 0.05,
                "Morning to Noon": 0.35,
                "Afternoon Hours": 0.30,
                "Night Time": 0.30
            }
        incident_time_of_day = random.choices(list(time_of_day_probs.keys()), list(time_of_day_probs.values()))[0]
        
        # NUMBER_OF_VEHICLES_INVOLVED
        if incident_type in ["Vehicle Theft", "Parked Car"]:
            number_of_vehicles_involved = None
        elif incident_type == "Single Vehicle Collision":
            number_of_vehicles_involved = 1
        else:
            number_of_vehicles_involved_probs = {
                2: 0.45,
                3: 0.30,
                4: 0.25
            }
            number_of_vehicles_involved = random.choices(list(number_of_vehicles_involved_probs.keys()), list(number_of_vehicles_involved_probs.values()))[0]
        
        # PROPERTY_DAMAGE
        if incident_type == "Multi Vehicle Collision":
            property_damage = "Property Damage"
        elif incident_type == "Single Vehicle Collision":
            property_damage_probs = {
                "Property Damage": 0.35,
                "No Property Damage": 0.65
            }
            property_damage = random.choices(list(property_damage_probs.keys()), list(property_damage_probs.values()))[0]
        else:
            property_damage = "No Property Damage"
        
        # INJURY_DAMAGE
        if authorities_contacted in ["Ambulance", "Fire"] or incident_type == "Multi Vehicle Collision":
            injury_damage = "Injury Damage"
        elif incident_type == "Single Vehicle Collision":
            injury_damage_probs = {
                "Injury Damage": 0.55,
                "No Injury Damage": 0.45
            }
            injury_damage = random.choices(list(injury_damage_probs.keys()), list(injury_damage_probs.values()))[0]
        else:
            injury_damage = "No Injury Damage"
        
        claims_data.append([
            claims_number, incident_date, policy_number, claim_reporting_date, incident_type, collision_type,
            loss_type, authorities_contacted, incident_state, incident_city, incident_county, incident_time_of_day,
            number_of_vehicles_involved, property_damage, injury_damage
        ])
    
    columns = [
        "CLAIMS_NUMBER", "INCIDENT_DATE", "POLICY_NUMBER", "CLAIM_REPORTING_DATE", "INCIDENT_TYPE", "COLLISION_TYPE",
        "LOSS_TYPE", "AUTHORITIES_CONTACTED", "INCIDENT_STATE", "INCIDENT_CITY", "INCIDENT_COUNTY", "INCIDENT_TIME_OF_DAY",
        "NUMBER_OF_VEHICLES_INVOLVED", "PROPERTY_DAMAGE", "INJURY_DAMAGE"
    ]
    
    return pd.DataFrame(claims_data, columns=columns)

# Number of records to generate
num_records = int(len(auto_insurance_policy_master) * 0.2)


# Generate the data
claims_data = generate_claims_data(auto_insurance_policy_master, num_records)

# Save the generated data to a CSV file
claims_data.to_csv('C:/Users/10741867/OneDrive - LTIMindtree/Documents/Datasets/Solution Datasets/Solution 1 - PNC Quote Conversion and Prioritization/Datasets/Regenerated Datasets/AUTO_INSURANCE_CLAIMS_MASTER_BASE.csv', index=False)

print("Synthetic data generation complete.")



Generating Claims Data: 100%|██████████| 190704/190704 [10:51:31<00:00,  4.88it/s]  


Synthetic data generation complete.
